## GINI Index


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


#### Step 1 — Load the Dataset

In [ ]:
df = pd.read_csv("messi_matches.csv")
print("Shape:", df.shape)
df


## Step 2 — Encode the Target Column
We convert **ManOfMatch** to numbers so we can do maths on it.  
`Yes = 1`, `No = 0`


In [ ]:
df["Label"] = df["ManOfMatch"].map({"Yes": 1, "No": 0})
print(df["Label"].value_counts())
df[["ManOfMatch", "Label"]]


## Step 3 — The Gini Index Formula

$$Gini = 1 - \sum_{i} p_i^2$$

Where $p_i$ is the proportion of each class in the group.

A **Gini of 0** means perfectly pure (all one class).  
A **Gini of 0.5** means maximally impure (50/50 split).


## Step 4 — Gini of the Whole Dataset (before any split)

In [ ]:
total = len(df)

count_yes = (df["Label"] == 1).sum()
count_no  = (df["Label"] == 0).sum()

p_yes = count_yes / total
p_no  = count_no  / total

gini_total = 1 - (p_yes**2 + p_no**2)

print("Total rows    :", total)
print("Yes (1)       :", count_yes)
print("No  (0)       :", count_no)
print("p_yes         :", round(p_yes, 3))
print("p_no          :", round(p_no,  3))
print()
print("Gini (whole)  =", round(gini_total, 4))


## Step 5 — Weighted Gini for Each Feature

For each feature we:
1. Split the data by that feature's values
2. Compute Gini inside each group
3. Take a **weighted average** (weighted by group size)

Let's do **Fitness** first as an example.


In [ ]:
feature = "Fitness"

groups = df.groupby(feature)["Label"]

weighted_gini = 0

for name, group in groups:
    n_group = len(group)
    weight  = n_group / total

    p1 = (group == 1).sum() / n_group
    p0 = (group == 0).sum() / n_group

    gini_group = 1 - (p1**2 + p0**2)

    print(f"  {feature} = {name}")
    print(f"    n={n_group}, p_yes={round(p1,3)}, p_no={round(p0,3)}, Gini={round(gini_group,4)}")
    print()

    weighted_gini += weight * gini_group

print("Weighted Gini for Fitness:", round(weighted_gini, 4))


## Step 6 — Weighted Gini for ALL Features

In [ ]:
features = ["Scored", "Assisted", "MatchType", "Fitness"]

gini_results = {}

for feature in features:

    groups = df.groupby(feature)["Label"]
    weighted_gini = 0

    for name, group in groups:
        n_group = len(group)
        weight  = n_group / total
        p1 = (group == 1).sum() / n_group
        p0 = (group == 0).sum() / n_group
        gini_group = 1 - (p1**2 + p0**2)
        weighted_gini += weight * gini_group

    gini_results[feature] = round(weighted_gini, 4)
    print(f"{feature:12s}  Weighted Gini = {round(weighted_gini, 4)}")

print()
best_feature = min(gini_results, key=gini_results.get)
print("Best feature to split on:", best_feature, "(lowest Gini)")


## Step 7 — Plot: Weighted Gini per Feature

In [ ]:
plt.figure(figsize=(8, 5))

feature_names = list(gini_results.keys())
gini_values   = list(gini_results.values())

bar_colors = ["#a50044" if f == best_feature else "#004d98" for f in feature_names]

plt.bar(feature_names, gini_values, color=bar_colors, edgecolor="black", width=0.5)

plt.title("Weighted Gini Index per Feature", fontsize=14, fontweight="bold")
plt.xlabel("Feature", fontsize=12)
plt.ylabel("Weighted Gini", fontsize=12)
plt.ylim(0, 0.6)

for i, val in enumerate(gini_values):
    plt.text(i, val + 0.01, str(val), ha="center", fontsize=11)

plt.axhline(y=gini_total, color="gray", linestyle="--", label=f"Original Gini = {round(gini_total,4)}")
plt.legend()
plt.tight_layout()
plt.savefig("gini_per_feature.png", dpi=120)
plt.show()
print("Red bar = best split feature")


## Step 8 — Gini Reduction (Impurity Drop)

$$\text{Gini Reduction} = Gini_{\text{before}} - Gini_{\text{weighted after split}}$$

The higher the reduction, the more that feature cleans up the data.


In [ ]:
reductions = {}

for feature, wg in gini_results.items():
    reduction = round(gini_total - wg, 4)
    reductions[feature] = reduction
    print(f"{feature:12s}  Gini Reduction = {reduction}")

print()
print("Best feature (highest reduction):", max(reductions, key=reductions.get))


## Step 9 — Heatmap: ManOfMatch by Best Feature (Fitness)

In [ ]:
cross = pd.crosstab(df["Fitness"], df["ManOfMatch"])
print(cross)
print()

plt.figure(figsize=(6, 4))
sns.heatmap(cross, annot=True, fmt="d", cmap="Blues", linewidths=0.5, cbar=True)
plt.title("ManOfMatch counts by Fitness Level", fontsize=13, fontweight="bold")
plt.xlabel("Man of Match")
plt.ylabel("Fitness")
plt.tight_layout()
plt.savefig("heatmap_fitness.png", dpi=120)
plt.show()


## Step 10 — Plot: Gini Reduction per Feature

In [ ]:
plt.figure(figsize=(8, 5))

feat_names = list(reductions.keys())
red_values = list(reductions.values())

bar_colors2 = ["#a50044" if f == best_feature else "#004d98" for f in feat_names]

plt.bar(feat_names, red_values, color=bar_colors2, edgecolor="black", width=0.5)

plt.title("Gini Reduction per Feature (Higher = Better)", fontsize=14, fontweight="bold")
plt.xlabel("Feature", fontsize=12)
plt.ylabel("Gini Reduction", fontsize=12)

for i, val in enumerate(red_values):
    plt.text(i, val + 0.005, str(val), ha="center", fontsize=11)

plt.tight_layout()
plt.savefig("gini_reduction.png", dpi=120)
plt.show()


## Summary

| Feature | Weighted Gini | Gini Reduction |
|---------|--------------|---------------|
| Scored | — | — |
| Assisted | — | — |
| MatchType | — | — |
| Fitness | — | — |

> Fill in from your output above!

**Conclusion:**  
The feature with the **lowest Weighted Gini** (= highest Gini Reduction) is the best first split for our Decision Tree.  
In our Messi dataset, **Fitness** wins — High fitness → Man of the Match. Low fitness → Not.
